In [2]:
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(
     accept ='',
    multiple = False
)
display(upload)
    

FileUpload(value=(), description='Upload')

In [3]:
! pip install kaggle

import os

os.makedirs('/root/.kaggle',exist_ok=True)
!mv kaggle.json/root/.kaggle/
! chmod 600 / root/.kaggle/kaggle.josn

mv: missing destination file operand after 'kaggle.json/root/.kaggle/'
Try 'mv --help' for more information.
chmod: changing permissions of '/': Permission denied
chmod: cannot access 'root/.kaggle/kaggle.josn': No such file or directory


In [4]:
!kaggle datasets download -d shahriarkabir/procurement-kpi-analysis-dataset

Dataset URL: https://www.kaggle.com/datasets/shahriarkabir/procurement-kpi-analysis-dataset
License(s): CC0-1.0
procurement-kpi-analysis-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
print(os.getcwd())

C:\Users\khushi jaiswal\Python days


In [6]:
import zipfile

with zipfile.ZipFile('procurement-kpi-analysis-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

In [8]:
os.listdir('data')

['Procurement KPI Analysis Dataset.csv', 'procurement_enriched.csv']

In [9]:
import pandas as pd

df = pd.read_csv('data/Procurement KPI Analysis Dataset.csv')
df.head()

C:\Users\khushi jaiswal\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,Yes
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,Yes
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,Yes
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,Yes
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,No


In [10]:
print(df.shape)
print(df.info())
df.describe()

(777, 11)
<class 'pandas.DataFrame'>
RangeIndex: 777 entries, 0 to 776
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PO_ID             777 non-null    str    
 1   Supplier          777 non-null    str    
 2   Order_Date        777 non-null    str    
 3   Delivery_Date     690 non-null    str    
 4   Item_Category     777 non-null    str    
 5   Order_Status      777 non-null    str    
 6   Quantity          777 non-null    int64  
 7   Unit_Price        777 non-null    float64
 8   Negotiated_Price  777 non-null    float64
 9   Defective_Units   641 non-null    float64
 10  Compliance        777 non-null    str    
dtypes: float64(3), int64(1), str(7)
memory usage: 113.6 KB
None


,Quantity,Unit_Price,Negotiated_Price,Defective_Units
count,777.000000,777.000000,777.000000,641.000000
mean,1094.660232,58.283822,53.660721,74.803432
std,647.844551,28.101315,26.094930,69.187870
min,51.000000,10.840000,9.270000,0.000000
25%,615.000000,33.290000,30.460000,26.000000
50%,1075.000000,58.950000,53.800000,49.000000
75%,1548.000000,83.130000,76.550000,100.000000
max,5000.000000,109.170000,107.390000,321.000000


In [11]:
print(df['Compliance'].unique())
print(df['Compliance'].value_counts())

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Compliance
Yes    640
No     137
Name: count, dtype: int64


In [12]:
df['Price_Gap_Pct'] = ((df['Unit_Price'] - df['Negotiated_Price']) / df['Unit_Price']) * 100
df[['Unit_Price', 'Negotiated_Price', 'Price_Gap_Pct']].head()

,Unit_Price,Negotiated_Price,Price_Gap_Pct
0,20.13,17.81,11.525087
1,39.32,37.34,5.035605
2,95.51,92.26,3.402785
3,99.85,95.52,4.336505
4,64.07,60.53,5.525207


In [13]:
df['Invoice_Number'] = ['INV' + str(1000 + i) for i in range(len(df))]
df[['PO_ID', 'Invoice_Number']].head()

,PO_ID,Invoice_Number
0,PO-00001,INV1000
1,PO-00002,INV1001
2,PO-00003,INV1002
3,PO-00004,INV1003
4,PO-00005,INV1004


In [14]:
import numpy as np

df['Order_Date'] = pd.to_datetime(df['Order_Date'])
np.random.seed(42)
random_days = np.random.randint(15, 45, size=len(df))
df['Payment_Date'] = df['Order_Date'] + pd.to_timedelta(random_days, unit='D')
df[['Order_Date', 'Payment_Date']].head()

,Order_Date,Payment_Date
0,2023-10-17,2023-11-07
1,2022-04-25,2022-05-29
2,2022-01-26,2022-03-10
3,2022-10-09,2022-11-07
4,2022-09-08,2022-10-03


In [15]:
df['is_fraud'] = 0

In [16]:
np.random.seed(1)
duplicate_indices = np.random.choice(df.index, size=15, replace=False)

for idx in duplicate_indices:
    # ek random doosri row ka invoice number copy kar do
    copy_from = np.random.choice(df.index[df.index != idx])
    df.loc[idx, 'Invoice_Number'] = df.loc[copy_from, 'Invoice_Number']
    df.loc[idx, 'is_fraud'] = 1

In [17]:
np.random.seed(2)
price_fraud_indices = np.random.choice(df.index[df['is_fraud']==0], size=20, replace=False)

df.loc[price_fraud_indices, 'Unit_Price'] = df.loc[price_fraud_indices, 'Unit_Price'] * np.random.uniform(2.5, 4, size=20)
df.loc[price_fraud_indices, 'Price_Gap_Pct'] = ((df.loc[price_fraud_indices, 'Unit_Price'] - df.loc[price_fraud_indices, 'Negotiated_Price']) / df.loc[price_fraud_indices, 'Unit_Price']) * 100
df.loc[price_fraud_indices, 'is_fraud'] = 1

In [18]:
np.random.seed(3)
payment_fraud_indices = np.random.choice(df.index[df['is_fraud']==0], size=15, replace=False)

# payment order date se bahut jaldi (2-3 din) ya bahut der (90+ din) kar do
odd_days = np.random.choice([2, 95, 100, 110], size=15)
df.loc[payment_fraud_indices, 'Payment_Date'] = df.loc[payment_fraud_indices, 'Order_Date'] + pd.to_timedelta(odd_days, unit='D')
df.loc[payment_fraud_indices, 'is_fraud'] = 1

In [19]:
np.random.seed(4)
suspicious_mask = (df['Compliance'] == 'No') & (df['Quantity'] > df['Quantity'].quantile(0.75)) & (df['is_fraud']==0)
suspicious_indices = df[suspicious_mask].sample(n=min(10, suspicious_mask.sum()), random_state=4).index

df.loc[suspicious_indices, 'is_fraud'] = 1

In [20]:
print(df['is_fraud'].value_counts())
print(f"Fraud %: {df['is_fraud'].mean()*100:.2f}%")

is_fraud
0    717
1     60
Name: count, dtype: int64
Fraud %: 7.72%


In [21]:
df.to_csv('data/procurement_enriched.csv', index=False)
print("Saved successfully!")
print(df.shape)

Saved successfully!
(777, 15)


OSError: Cannot save file into a non-existent directory: 'D:\dataset'

In [22]:
df.to_csv(r"D:\Data set\procurement_enriched.csv", index=False)
print("Saved successfully!")

Saved successfully!


In [24]:
import pandas as pd

df = pd.read_csv(r"D:\Data set\procurement_enriched.csv")
df.head()

,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance,Price_Gap_Pct,Invoice_Number,Payment_Date,is_fraud
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,Yes,11.525087,INV1000,2023-11-07,0
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,Yes,5.035605,INV1001,2022-05-29,0
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,Yes,3.402785,INV1002,2022-03-10,0
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,Yes,4.336505,INV1003,2022-11-07,0
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,No,5.525207,INV1004,2022-10-03,0


In [25]:
df['Defective_Units'] = df['Defective_Units'].fillna(df['Defective_Units'].median())

In [27]:
features = ['Quantity', 'Unit_Price', 'Negotiated_Price', 'Price_Gap_Pct', 'Defective_Units']
X = df[features]

In [28]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(contamination=0.08, random_state=42)
model.fit(X)

df['anomaly_score'] = model.decision_function(X)
df['ml_flag'] = model.predict(X)   # -1 = anomaly (suspicious), 1 = normal

In [29]:
print(df['ml_flag'].value_counts())

ml_flag
 1    714
-1     63
Name: count, dtype: int64


In [30]:
# Kitna overlap hai ML detection aur humare planted fraud me
comparison = pd.crosstab(df['is_fraud'], df['ml_flag'])
print(comparison)

ml_flag   -1    1
is_fraud         
0         37  680
1         26   34


In [31]:
df.to_csv(r"D:\Data set\procurement_final_with_ml.csv", index=False)
print("Final file saved for Power BI!")

Final file saved for Power BI!
